# 🧩 Mini-Lab: Multi-Document Type Extraction

**Module 3: Structured Extraction** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Classify** documents by type automatically
2. **Select** the appropriate schema for each document type
3. **Build** a universal extractor that handles any document
4. **Process** all four document types: invoices, receipts, resumes, contracts

## Target Concepts

| Concept | Description |
|---------|-------------|
| Document classification | Using LLM to automatically identify document type before extraction |
| Schema selection | Mapping document types to appropriate Pydantic schemas |
| Universal extractor | Single interface that handles multiple document types automatically |
| Type-specific extraction | Applying different extraction logic based on document classification |

In [21]:
import json
from pathlib import Path
from openai import OpenAI
from pydantic import BaseModel
from dotenv import load_dotenv
from IPython.display import Markdown, display

# Import our schemas
from document_schemas import (
    Invoice, Receipt, Resume, Contract,
    DOCUMENT_SCHEMAS, DOCUMENT_TYPES
)

def md(text):
    display(Markdown(text))

load_dotenv()
client = OpenAI()

DATA_DIR = Path('../data/')

print('✓ Setup complete')
print(f'✓ Document types: {DOCUMENT_TYPES}')

✓ Setup complete
✓ Document types: ['invoice', 'receipt', 'resume', 'contract']


---
## 1. Load All Document Types

Let's load samples from each document type.

In [22]:
def load_documents(folder_name: str, limit: int = 2) -> dict[str, str]:
    """Load documents from a folder."""
    folder = DATA_DIR / folder_name
    docs = {}
    for file in sorted(folder.glob('*.txt'))[:limit]:
        docs[file.stem] = file.read_text(encoding='utf-8')
    return docs

# Load samples from each type
all_documents = {
    'invoice': load_documents('invoices', 2),
    'receipt': load_documents('receipts', 2),
    'resume': load_documents('resumes', 2),
    'contract': load_documents('contracts', 2)
}

print('📁 Loaded documents:')
for doc_type, docs in all_documents.items():
    print(f'  {doc_type}: {list(docs.keys())}')

📁 Loaded documents:
  invoice: ['invoice_001_corporate', 'invoice_002_simple_receipt']
  receipt: ['receipt_001_retail', 'receipt_002_restaurant']
  resume: ['resume_001_software_engineer', 'resume_002_recent_graduate']
  contract: ['contract_001_employment', 'contract_002_service']


---
## 2. Document Classification

Before extracting, we need to identify what type of document we're dealing with.

In [23]:
def classify_document(document: str) -> str:
    """
    Classify a document into one of the known types.
    
    Args:
        document: The document text
        
    Returns:
        Document type: 'invoice', 'receipt', 'resume', or 'contract'
    """
    system_prompt = f'''You are a document classifier.
Classify the document into exactly one of these types: {', '.join(DOCUMENT_TYPES)}.

Guidelines:
- invoice: Bills for services/products with line items, totals, payment terms
- receipt: Proof of purchase from a store, restaurant, or online order
- resume: Job application document with experience, education, skills
- contract: Legal agreement between parties with terms and signatures

Respond with ONLY the document type, nothing else.'''

    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': document[:2000]}  # Limit for speed
        ],
        temperature=0
    )
    
    doc_type = response.choices[0].message.content.strip().lower()
    
    # Validate
    if doc_type in DOCUMENT_TYPES:
        return doc_type
    
    # Fallback: try to match
    for dt in DOCUMENT_TYPES:
        if dt in doc_type:
            return dt
    
    return 'invoice'  # Default

# Test classification on each type
print('🔍 Testing document classification:\n')

for expected_type, docs in all_documents.items():
    for name, content in list(docs.items())[:1]:  # Test first of each
        predicted = classify_document(content)
        status = '✅' if predicted == expected_type else '❌'
        print(f'{status} {name}: expected={expected_type}, predicted={predicted}')

🔍 Testing document classification:

✅ invoice_001_corporate: expected=invoice, predicted=invoice
✅ receipt_001_retail: expected=receipt, predicted=receipt
✅ resume_001_software_engineer: expected=resume, predicted=resume
✅ contract_001_employment: expected=contract, predicted=contract


---
## 3. Schema-Based Extraction

Use the classified type to select the right schema.

In [24]:
def extract_with_schema(document: str, schema: type[BaseModel]) -> BaseModel:
    """
    Extract data using a specific Pydantic schema.
    """
    schema_json = json.dumps(schema.model_json_schema(), indent=2)
    
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {
                'role': 'system',
                'content': f'''Extract data according to this JSON schema:
{schema_json}

Return only valid JSON. Use null for missing fields.'''
            },
            {'role': 'user', 'content': f'Document:\n{document}'}
        ],
        response_format={'type': 'json_object'},
        temperature=0
    )
    
    data = json.loads(response.choices[0].message.content)
    return schema(**data)


# Test extraction for each document type
print('📤 Testing schema-based extraction:\n')

📤 Testing schema-based extraction:



In [25]:
# Extract an invoice
sample_invoice = list(all_documents['invoice'].values())[1]
invoice_result = extract_with_schema(sample_invoice, Invoice)

print('📋 INVOICE EXTRACTION:')
print(f'  Number: {invoice_result.invoice_number}')
print(f'  Vendor: {invoice_result.vendor_name}')
print(f'  Customer: {invoice_result.customer_name}')
print(f'  Total: ${invoice_result.total:,.2f}')

📋 INVOICE EXTRACTION:
  Number: REC-8847
  Vendor: CloudHost Services
  Customer: DataFlow Analytics
  Total: $406.00


In [26]:
# Extract a receipt
sample_receipt = list(all_documents['receipt'].values())[0]
receipt_result = extract_with_schema(sample_receipt, Receipt)

print('🧾 RECEIPT EXTRACTION:')
print(f'  Number: {receipt_result.receipt_number}')
print(f'  Store: {receipt_result.store_name}')
print(f'  Items: {len(receipt_result.items)}')
print(f'  Total: ${receipt_result.total:,.2f}')
print(f'  Payment: {receipt_result.payment_method}')

🧾 RECEIPT EXTRACTION:
  Number: TXN-20240322-2847-12
  Store: BEST BUY ELECTRONICS
  Items: 6
  Total: $2,667.95
  Payment: Visa ****4521


In [27]:
# Extract a resume
sample_resume = list(all_documents['resume'].values())[0]
resume_result = extract_with_schema(sample_resume, Resume)

print('📄 RESUME EXTRACTION:')
print(f'  Name: {resume_result.full_name}')
print(f'  Email: {resume_result.email}')
print(f'  Location: {resume_result.location}')
print(f'  Experience: {len(resume_result.experience)} positions')
print(f'  Skills: {len(resume_result.skills)} skills')
print(f'  Education: {len(resume_result.education)} entries')

📄 RESUME EXTRACTION:
  Name: ALEXANDER CHEN
  Email: alex.chen@email.com
  Location: San Francisco, CA
  Experience: 3 positions
  Skills: 26 skills
  Education: 1 entries


In [28]:
# Extract a contract
sample_contract = list(all_documents['contract'].values())[0]
contract_result = extract_with_schema(sample_contract, Contract)

print('📜 CONTRACT EXTRACTION:')
print(f'  Title: {contract_result.contract_title}')
print(f'  Type: {contract_result.contract_type}')
print(f'  Effective: {contract_result.effective_date}')
print(f'  Parties: {len(contract_result.parties)}')
print(f'  Value: ${contract_result.total_value:,.2f}' if contract_result.total_value else '  Value: N/A')
print(f'  Governing Law: {contract_result.governing_law}')

📜 CONTRACT EXTRACTION:
  Title: Senior Software Engineer Employment Agreement
  Type: Employment
  Effective: 2024-04-01
  Parties: 2
  Value: $150,000.00
  Governing Law: State of California


---
## 4. Universal Extractor

Combine classification and extraction into a single function.

In [29]:
def universal_extract(document: str) -> tuple[str, BaseModel]:
    """
    Automatically detect document type and extract data.
    
    Args:
        document: Any document text
        
    Returns:
        Tuple of (document_type, extracted_data)
    """
    # Step 1: Classify
    doc_type = classify_document(document)
    
    # Step 2: Get schema
    schema = DOCUMENT_SCHEMAS[doc_type]
    
    # Step 3: Extract
    extracted = extract_with_schema(document, schema)
    
    return doc_type, extracted


# Test universal extractor
print('🔄 Universal Extractor Test:\n')

# Pick random samples
test_docs = [
    list(all_documents['invoice'].values())[0],
    list(all_documents['receipt'].values())[0],
    list(all_documents['resume'].values())[0],
    list(all_documents['contract'].values())[0]
]

for doc in test_docs:
    doc_type, result = universal_extract(doc)
    
    # Get a summary field based on type
    if doc_type == 'invoice':
        summary = f"{result.vendor_name} → {result.customer_name}: ${result.total:,.2f}"
    elif doc_type == 'receipt':
        summary = f"{result.store_name}: ${result.total:,.2f}"
    elif doc_type == 'resume':
        summary = f"{result.full_name} ({result.email})"
    else:  # contract
        summary = f"{result.contract_title} ({result.contract_type})"
    
    print(f'📄 {doc_type.upper()}: {summary}')

🔄 Universal Extractor Test:



📄 INVOICE: TechCorp Solutions Inc. → Acme Corporation: $24,686.46
📄 RECEIPT: BEST BUY ELECTRONICS: $2,667.95
📄 RESUME: ALEXANDER CHEN (alex.chen@email.com)
📄 CONTRACT: Senior Software Engineer Employment Agreement (Employment)


---
## 5. Processing All Documents

Let's process all documents and create a summary report.

In [30]:
def process_all_documents():
    """Process all documents and return results."""
    results = []
    
    for expected_type, docs in all_documents.items():
        for name, content in docs.items():
            print(f'Processing: {name}...')
            
            try:
                detected_type, extracted = universal_extract(content)
                results.append({
                    'file': name,
                    'expected_type': expected_type,
                    'detected_type': detected_type,
                    'correct': expected_type == detected_type,
                    'data': extracted,
                    'error': None
                })
            except Exception as e:
                results.append({
                    'file': name,
                    'expected_type': expected_type,
                    'detected_type': None,
                    'correct': False,
                    'data': None,
                    'error': str(e)
                })
    
    return results

# Process all
results = process_all_documents()
print('\n✓ Processing complete!')

Processing: invoice_001_corporate...


Processing: invoice_002_simple_receipt...
Processing: receipt_001_retail...
Processing: receipt_002_restaurant...
Processing: resume_001_software_engineer...
Processing: resume_002_recent_graduate...
Processing: contract_001_employment...
Processing: contract_002_service...

✓ Processing complete!


In [31]:
# Summary table
md('## Results Summary\n')

print('| File | Expected | Detected | Status |')
print('|------|----------|----------|--------|')

correct = 0
total = len(results)

for r in results:
    status = '✅' if r['correct'] else '❌'
    if r['correct']:
        correct += 1
    print(f"| {r['file'][:25]} | {r['expected_type']} | {r['detected_type']} | {status} |")

print(f'\n📊 Accuracy: {correct}/{total} ({100*correct/total:.1f}%)')

## Results Summary


| File | Expected | Detected | Status |
|------|----------|----------|--------|
| invoice_001_corporate | invoice | invoice | ✅ |
| invoice_002_simple_receip | invoice | invoice | ✅ |
| receipt_001_retail | receipt | receipt | ✅ |
| receipt_002_restaurant | receipt | receipt | ✅ |
| resume_001_software_engin | resume | resume | ✅ |
| resume_002_recent_graduat | resume | resume | ✅ |
| contract_001_employment | contract | contract | ✅ |
| contract_002_service | contract | contract | ✅ |

📊 Accuracy: 8/8 (100.0%)


---
## 6. Detailed Results by Type

In [32]:
# Show detailed results for each type
for doc_type in DOCUMENT_TYPES:
    type_results = [r for r in results if r['expected_type'] == doc_type and r['data']]
    
    if not type_results:
        continue
    
    print(f'\n{"="*60}')
    print(f'{doc_type.upper()} EXTRACTIONS')
    print('='*60)
    
    for r in type_results:
        data = r['data']
        print(f'\n📄 {r["file"]}:')
        
        if doc_type == 'invoice':
            print(f'   Invoice #: {data.invoice_number}')
            print(f'   Vendor: {data.vendor_name}')
            print(f'   Total: ${data.total:,.2f}')
        
        elif doc_type == 'receipt':
            print(f'   Receipt #: {data.receipt_number}')
            print(f'   Store: {data.store_name}')
            print(f'   Total: ${data.total:,.2f}')
        
        elif doc_type == 'resume':
            print(f'   Name: {data.full_name}')
            print(f'   Email: {data.email}')
            print(f'   Skills: {len(data.skills)} listed')
        
        elif doc_type == 'contract':
            print(f'   Title: {data.contract_title}')
            print(f'   Type: {data.contract_type}')
            print(f'   Law: {data.governing_law}')


INVOICE EXTRACTIONS

📄 invoice_001_corporate:
   Invoice #: INV-2024-0842
   Vendor: TechCorp Solutions Inc.
   Total: $24,686.46

📄 invoice_002_simple_receipt:
   Invoice #: REC-8847
   Vendor: CloudHost Services
   Total: $406.00

RECEIPT EXTRACTIONS

📄 receipt_001_retail:
   Receipt #: TXN-20240322-2847-12
   Store: BEST BUY ELECTRONICS
   Total: $2,667.95

📄 receipt_002_restaurant:
   Receipt #: 2847
   Store: THE GARDEN BISTRO
   Total: $162.33

RESUME EXTRACTIONS

📄 resume_001_software_engineer:
   Name: ALEXANDER CHEN
   Email: alex.chen@email.com
   Skills: 26 listed

📄 resume_002_recent_graduate:
   Name: JESSICA MARTINEZ
   Email: jessica.martinez@email.com
   Skills: 21 listed

CONTRACT EXTRACTIONS

📄 contract_001_employment:
   Title: Senior Software Engineer Employment Agreement
   Type: Employment
   Law: State of California

📄 contract_002_service:
   Title: Custom Web Application Development Services
   Type: Service
   Law: State of California


## 🎯 Summary

### Key Takeaways

1. **Document classification** — LLM can reliably identify document types when given clear definitions
2. **Schema mapping** — maintain a dictionary that maps document types to their Pydantic schemas
3. **Universal extractor pattern** — classify first, then select schema, then extract with validation
4. **Type-specific handling** — different document types require different schemas and extraction approaches

---
## Next Steps

In the next notebook, we'll learn how to:
- **Calculate confidence scores** using log probabilities
- **Handle extraction failures** gracefully
- **Implement retry strategies** with fallbacks

→ Continue to **mini-doc-extractor-04-error-handling.ipynb**